In [1]:
"""
============================
Tutorial 0: Getting Started
============================

This tutorial takes you through a basic working example of how to use this
codebase, including all the different components, up to the results
generation. If you'd like to know about the statistics and plotting, see the
next tutorial.

"""

# Authors: Vinay Jayaram <vinayjayaram13@gmail.com>
#
# License: BSD (3-clause)


##########################################################################
# Introduction
# --------------------
# To use the codebase you need an evaluation and a paradigm, some algorithms,
# and a list of datasets to run it all on. You can find those in the following
# submodules; detailed tutorials are given for each of them.

import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC


In [2]:

##########################################################################
# If you would like to specify the logging level when it is running, you can
# use the standard python logging commands through the top-level moabb module
import moabb
from moabb.datasets import BNCI2014_001, utils
from moabb.evaluations import CrossSessionEvaluation
from moabb.paradigms import LeftRightImagery
from moabb.pipelines.features import LogVariance


##########################################################################
# In order to create pipelines within a script, you will likely need at least
# the make_pipeline function. They can also be specified via a .yml file. Here
# we will make a couple pipelines just for convenience


moabb.set_log_level("info")


In [3]:

##############################################################################
# Create pipelines
# ----------------
#
# We create two pipelines: channel-wise log variance followed by LDA, and
# channel-wise log variance followed by a cross-validated SVM (note that a
# cross-validation via scikit-learn cannot be described in a .yml file). For
# later in the process, the pipelines need to be in a dictionary where the key
# is the name of the pipeline and the value is the Pipeline object

pipelines = {}
pipelines["AM+LDA"] = make_pipeline(LogVariance(), LDA())
parameters = {"C": np.logspace(-2, 2, 10)}
clf = GridSearchCV(SVC(kernel="linear"), parameters)
pipe = make_pipeline(LogVariance(), clf)

pipelines["AM+SVM"] = pipe

In [4]:
##############################################################################
# Datasets
# -----------------
#
# Datasets can be specified in many ways: Each paradigm has a property
# 'datasets' which returns the datasets that are appropriate for that paradigm

print(LeftRightImagery().datasets)


[<moabb.datasets.bnci.BNCI2014_001 object at 0x134491550>, <moabb.datasets.bnci.BNCI2014_004 object at 0x134491940>, <moabb.datasets.beetl.Beetl2021_A object at 0x134491a90>, <moabb.datasets.beetl.Beetl2021_B object at 0x1344916a0>, <moabb.datasets.gigadb.Cho2017 object at 0x134491be0>, <moabb.datasets.dreyer2023.Dreyer2023 object at 0x134491e80>, <moabb.datasets.dreyer2023.Dreyer2023A object at 0x134491d30>, <moabb.datasets.dreyer2023.Dreyer2023B object at 0x134491fd0>, <moabb.datasets.dreyer2023.Dreyer2023C object at 0x134492120>, <moabb.datasets.mpi_mi.GrosseWentrup2009 object at 0x134492270>, <moabb.datasets.Lee2019.Lee2019_MI object at 0x134492510>, <moabb.datasets.liu2024.Liu2024 object at 0x13035d160>, <moabb.datasets.physionet_mi.PhysionetMI object at 0x1344923c0>, <moabb.datasets.schirrmeister2017.Schirrmeister2017 object at 0x13035d010>, <moabb.datasets.bbci_eeg_fnirs.Shin2017A object at 0x134492660>, <moabb.datasets.stieger2021.Stieger2021 object at 0x130727e00>, <moabb.data

In [5]:
##########################################################################
# Or you can run a search through the available datasets:
print(utils.dataset_search(paradigm="imagery", min_subjects=6))

[<moabb.datasets.alex_mi.AlexMI object at 0x13033d810>, <moabb.datasets.bnci.BNCI2014_001 object at 0x1336b7750>, <moabb.datasets.bnci.BNCI2014_002 object at 0x13033d6d0>, <moabb.datasets.bnci.BNCI2014_004 object at 0x1336b74d0>, <moabb.datasets.bnci.BNCI2015_001 object at 0x1336b7610>, <moabb.datasets.bnci.BNCI2015_004 object at 0x1336b79d0>, <moabb.datasets.gigadb.Cho2017 object at 0x1336b7890>, <moabb.datasets.dreyer2023.Dreyer2023 object at 0x1336b7c50>, <moabb.datasets.dreyer2023.Dreyer2023A object at 0x1336b7b10>, <moabb.datasets.dreyer2023.Dreyer2023B object at 0x1336b7d90>, <moabb.datasets.dreyer2023.Dreyer2023C object at 0x1336b7ed0>, <moabb.datasets.fake.FakeDataset object at 0x1344e0050>, <moabb.datasets.mpi_mi.GrosseWentrup2009 object at 0x1344e02d0>, <moabb.datasets.Lee2019.Lee2019_MI object at 0x1344e0410>, <moabb.datasets.liu2024.Liu2024 object at 0x1344e0550>, <moabb.datasets.upper_limb.Ofner2017 object at 0x1344e0190>, <moabb.datasets.physionet_mi.PhysionetMI object at

In [6]:
##########################################################################
# Or you can simply make your own list (which we do here due to computational
# constraints)

dataset = BNCI2014_001()
dataset.subject_list = dataset.subject_list[:2]
datasets = [dataset]

In [7]:
##########################################################################
# Paradigm
# --------------------
#
# Paradigms define the events, epoch time, bandpass, and other preprocessing
# parameters. They have defaults that you can read in the documentation, or you
# can simply set them as we do here. A single paradigm defines a method for
# going from continuous data to trial data of a fixed size. To learn more look
# at the tutorial Exploring Paradigms

fmin = 8
fmax = 35
paradigm = LeftRightImagery(fmin=fmin, fmax=fmax)

In [8]:
##########################################################################
# Evaluation
# --------------------
#
# An evaluation defines how the training and test sets are chosen. This could
# be cross-validated within a single recording, or across days, or sessions, or
# subjects. This also is the correct place to specify multiple threads.

evaluation = CrossSessionEvaluation(
    paradigm=paradigm, datasets=datasets, suffix="examples", overwrite=False
)
results = evaluation.process(pipelines)

BNCI2014-001-CrossSession: 100%|██████████| 2/2 [00:05<00:00,  2.51s/it]
2025-12-18 09:00:47,984 INFO MainThread moabb.evaluations.base AM+LDA | BNCI2014-001 | 1 | 0train: Score 0.786
2025-12-18 09:00:48,041 INFO MainThread moabb.evaluations.base AM+LDA | BNCI2014-001 | 1 | 1test: Score 0.802
2025-12-18 09:00:48,100 INFO MainThread moabb.evaluations.base AM+SVM | BNCI2014-001 | 1 | 0train: Score 0.797
2025-12-18 09:00:48,161 INFO MainThread moabb.evaluations.base AM+SVM | BNCI2014-001 | 1 | 1test: Score 0.774
2025-12-18 09:00:48,221 INFO MainThread moabb.evaluations.base AM+LDA | BNCI2014-001 | 2 | 0train: Score 0.577
2025-12-18 09:00:48,276 INFO MainThread moabb.evaluations.base AM+LDA | BNCI2014-001 | 2 | 1test: Score 0.499
2025-12-18 09:00:48,332 INFO MainThread moabb.evaluations.base AM+SVM | BNCI2014-001 | 2 | 0train: Score 0.551
2025-12-18 09:00:48,396 INFO MainThread moabb.evaluations.base AM+SVM | BNCI2014-001 | 2 | 1test: Score 0.471


In [9]:
##########################################################################
# Results are returned as a pandas DataFrame, and from here you can do as you
# want with them

print(results.head())


      score      time  samples subject session  channels  n_sessions  \
0  0.786458  0.019537    144.0       1  0train        22           2   
1  0.801698  0.018425    144.0       1   1test        22           2   
2  0.576582  0.019189    144.0       2  0train        22           2   
3  0.498650  0.018031    144.0       2   1test        22           2   
4  0.797068  0.080576    144.0       1  0train        22           2   

        dataset pipeline  
0  BNCI2014-001   AM+LDA  
1  BNCI2014-001   AM+LDA  
2  BNCI2014-001   AM+LDA  
3  BNCI2014-001   AM+LDA  
4  BNCI2014-001   AM+SVM  
